In [2]:
!pip install gymnasium[atari] ale-py -q
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
import ale_py
import torch
import torch.nn as nn
import cv2

gym.register_envs(ale_py)

In [3]:
class DQN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Linear(3136, 512), nn.ReLU(),
            nn.Linear(512, 6),
        )
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DQN(n_actions=6).to(device)
model.load_state_dict(torch.load("/kaggle/input/models/joercharles/pongbot/pytorch/default/1/pong_dqn.pth", map_location=device))
model.eval()
print("Weights loaded successfully")

Weights loaded successfully


In [5]:
def make_env(env_name="ALE/Pong-v5"):
    env = gym.make(env_name, render_mode="rgb_array", frameskip=1)
    env = AtariPreprocessing(env)
    env = FrameStackObservation(env, 4)
    return env

In [6]:
def record_agent(net, n_episodes=10, output_path="/kaggle/working/pong_agent.avi"):
    env = make_env()
    obs, _ = env.reset()
    raw_frame = env.render()
    height, width, _ = raw_frame.shape

    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    writer = cv2.VideoWriter(output_path, fourcc, 30, (width, height))

    model = net.module if hasattr(net, "module") else net
    model.eval()

    for episode in range(n_episodes):
        obs, _ = env.reset()
        total_reward = 0.0
        done = False
        while not done:
            frame = env.render()
            writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
            with torch.no_grad():
                state_t = torch.tensor(np.array(obs), dtype=torch.float32).unsqueeze(0).to(device) / 255.0
                action = model(state_t).argmax(dim=1).item()
            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated
        print(f"Episode {episode + 1} | Reward: {total_reward:.1f}")

    writer.release()
    env.close()
    print(f"Video saved to {output_path}")

record_agent(model)

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


Episode 1 | Reward: 14.0
Episode 2 | Reward: 14.0
Episode 3 | Reward: 13.0
Episode 4 | Reward: 15.0
Episode 5 | Reward: 16.0
Episode 6 | Reward: 4.0
Episode 7 | Reward: 10.0
Episode 8 | Reward: 5.0
Episode 9 | Reward: 10.0
Episode 10 | Reward: 13.0
Video saved to /kaggle/working/pong_agent.avi
